In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/web_series_dataset.csv")

In [ ]:
df.shape

(55110, 4)

In [ ]:
df.head()

,title,overview,url,genre
0,Wednesday,"Smart, sarcastic and a little dead inside, Wed...",https://image.tmdb.org/t/p/w780/36xXlhEpQqVVPu...,"Sci-Fi & Fantasy, Mystery, Comedy"
1,The Summer I Turned Pretty,"Every summer, Belly and her family head to the...",https://image.tmdb.org/t/p/w780/xBIz53wYWsKfFp...,Drama
2,Alien: Earth,When the mysterious deep space research vessel...,https://image.tmdb.org/t/p/w780/yueXS3q8BtoWek...,"Sci-Fi & Fantasy, Drama"
3,Dexter,"Dexter Morgan, a blood spatter pattern analyst...",https://image.tmdb.org/t/p/w780/q8dWfc4JwQuv3H...,"Crime, Drama, Mystery"
4,Law & Order: Special Victims Unit,"In the criminal justice system, sexually-based...",https://image.tmdb.org/t/p/w780/34O6znqnLHAa19...,"Crime, Drama, Mystery"


In [ ]:
df.isnull().sum()

,0
title,0
overview,0
url,0
genre,9897


In [ ]:
df = df.dropna()

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00


In [ ]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
documents = ("Title:" + df['title'].astype(str) +
             "\nGenre:" + df['genre'].astype(str) +
             "\nOverview:" + df["overview"].astype(str)).tolist()

In [ ]:
embeddings = model.encode(documents,batch_size=1024,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=True)

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
dimension = embeddings.shape[1]

In [ ]:
index = faiss.IndexFlatIP(dimension)

In [ ]:
index.add(embeddings)

In [ ]:
index.ntotal

45213

In [ ]:
len(df)

45213

In [ ]:
np.save("/content/drive/MyDrive/webseries_embeddings.npy",embeddings)

In [ ]:
faiss.write_index(index,"/content/drive/MyDrive/webseries_index")
model.save("/content/drive/MyDrive//webseries")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print("Dataset rows:", len(df))
print("FAISS vectors:", index.ntotal)

print("\nFirst 10 dataset rows:")
print(df[["title", "genre", "overview"]].head(10))

print("\nTesting FAISS search...")

test_text = (
    "Title: Dark "
    "Genre: Mystery, Sci-Fi, Drama "
    "Overview: A family discovers a mysterious connection between "
    "different generations, time travel, and a small town hiding a dark secret."
)

embedding = model.encode(
    [test_text],
    normalize_embeddings=True
)

scores, indexes = index.search(embedding, 10)

print("\nFAISS Results:")

for rank, i in enumerate(indexes[0], start=1):

    series = df.iloc[i]

    print(
        rank,
        "| Index:", i,
        "| Score:", float(scores[0][rank - 1]),
        "| Title:", series["title"]
    )

Dataset rows: 45213
FAISS vectors: 45213

First 10 dataset rows:
                               title                                  genre  \
0                          Wednesday      Sci-Fi & Fantasy, Mystery, Comedy   
1         The Summer I Turned Pretty                                  Drama   
2                       Alien: Earth                Sci-Fi & Fantasy, Drama   
3                             Dexter                  Crime, Drama, Mystery   
4  Law & Order: Special Victims Unit                  Crime, Drama, Mystery   
5                         The Player                   Action, Crime, Drama   
6         How do you like Wednesday?                                 Comedy   
7                             Clerks                      Animation, Comedy   
8                    Shuriken School  Comedy, Animation, Action & Adventure   
9        Mister Rogers' Neighborhood                                   Kids   

                                            overview  
0  Smart, 

In [ ]:
df.to_csv("web_series_dataset.csv",index=False)